In [1]:
import os
from pathlib import Path
import numpy as np
import sys

# Add repo root so we can import utils
repo_root = Path("/home/cpanourg/projects/2-hdvc")
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.utils import write_fvecs

DATASET_DIR = Path("/mnthdd/cpanourg/2-hdvc/data/deep1b/dataset")
OUT_DIR = DATASET_DIR / "fvecs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIM = 96
CHUNK_VECS = 100_000  # adjust based on RAM


def bin_to_fvecs(bin_path: Path, out_path: Path, dim: int, chunk_vecs: int):
    # Detect header (fbin) vs raw float32
    file_size = bin_path.stat().st_size
    has_header = False
    nvecs = None
    offset = 0

    if (file_size - 8) % (4 * dim) == 0:
        # Potential fbin header
        header = np.fromfile(bin_path, dtype=np.int32, count=2)
        if header.size == 2 and header[1] == dim:
            has_header = True
            nvecs = int(header[0])
            offset = 8

    if not has_header:
        if file_size % (4 * dim) != 0:
            raise ValueError(f"File size not divisible by dim: {bin_path}")
        nvecs = file_size // (4 * dim)

    print(f"Converting {bin_path.name}: nvecs={nvecs}, dim={dim}, header={has_header}")

    # Memory-map the data to avoid loading all at once
    data = np.memmap(
        bin_path, dtype=np.float32, mode="r", offset=offset, shape=(nvecs, dim)
    )

    with open(out_path, "wb") as f:
        for start in range(0, nvecs, chunk_vecs):
            end = min(start + chunk_vecs, nvecs)
            chunk = np.asarray(data[start:end], dtype=np.float32)
            # Write as fvecs: [dim(int32) | vector(float32)] per row
            dim_col = np.full((chunk.shape[0], 1), dim, dtype=np.int32)
            out = np.concatenate([dim_col, chunk.view(np.int32)], axis=1)
            out.tofile(f)

    print(f"Saved: {out_path}")


for bin_file in ["learn_100m.bin", "test_1m.bin", "query_10k.bin"]:
    src = DATASET_DIR / bin_file
    dst = OUT_DIR / (src.stem + ".fvecs")
    bin_to_fvecs(src, dst, DIM, CHUNK_VECS)



Converting learn_100m.bin: nvecs=100000000, dim=96, header=False
Saved: /mnthdd/cpanourg/2-hdvc/data/deep1b/dataset/fvecs/learn_100m.fvecs
Converting test_1m.bin: nvecs=1000000, dim=96, header=False
Saved: /mnthdd/cpanourg/2-hdvc/data/deep1b/dataset/fvecs/test_1m.fvecs
Converting query_10k.bin: nvecs=10000, dim=96, header=False
Saved: /mnthdd/cpanourg/2-hdvc/data/deep1b/dataset/fvecs/query_10k.fvecs
